# Notebook 03 — Cluster Stability Analysis

**Research question**: Are the regimes identified by the Jump Model structurally coherent, or are they artefacts of the specific estimation window chosen?

Diagnostics are anchored on the **biannual refit calendar** (Jan / Jul), i.e. the same cadence as λ cross-validation in the main pipeline.

### Relative / transition metrics (no ground truth)

| Metric | Description | Ideal |
|--------|-------------|--------|
| **Label Agreement Rate** | Overlap days: agreement between **consecutive** refits | High (> 0.80) |
| **ARI** | Chance-corrected agreement between consecutive refits | High (> 0.60) |
| **Persistence** | Mean regime run length in the training window | Context-dependent |
| **Centroid Drift** | ‖Δ‖ between aligned bull / bear centroids | Low |
| **Occupancy std** | Dispersion of % bearish across refit windows | Low |

### Internal cluster validity (per refit)

At **each** rebalancing date, on the JM training matrix `X` and the fitted binary labels:

| Metric | Ideal | Note |
|--------|--------|------|
| **Silhouette** | ↑ | Primary scale-free geometry index |
| **Davies–Bouldin** | ↓ | Primary separation compactness index |
| **Calinski–Harabasz / n** | ↑ | CH scaled by sample size (raw CH is sensitive to *n* and is kept in the CSV for reference / Hypothesis B) |

For **temporal stability vs macro stress** (Notebook 04), we correlate only **Silhouette** and **Davies–Bouldin** with stress proxies. Raw CH is not used there.

These indices are **geometric** only (no external labels). They are reported **over time** in `stability_cvi_by_rebal.csv` and Fig. 4.

### Method
For each biannual date we fit the JM on the trailing window (fixed λ in this notebook). Pairwise metrics compare refit `t` vs the previous refit. CVIs characterise the partition at a single refit.

In [ ]:
import sys, os, pickle, warnings
from pathlib import Path

# Jupyter cwd varies (repo root, notebooks/, …): walk up until we find settings.py
_here = Path.cwd().resolve()
_repo_root = _here
for _ in range(16):
    if (_repo_root / "src" / "config" / "settings.py").is_file():
        break
    if _repo_root.parent == _repo_root:
        raise RuntimeError(
            "Cannot find thesis repo (src/config/settings.py). cwd=%r" % (_here,)
        )
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.utils.helpers import setup_logging
setup_logging()

RESULTS_DIR = _repo_root / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

from src.config.settings import (
    ASSETS, ASSET_TICKERS, FRED_SERIES,
    DATA_START, DATA_END, TEST_START, TEST_END,
    TRAIN_YEARS, REBAL_MONTHS, ASSETS_NO_DD_IN_JM,
)
from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor

loader = DataLoader()
prices = loader.load_prices(ASSET_TICKERS, start=DATA_START, end=DATA_END)
fred   = loader.load_fred(FRED_SERIES,     start=DATA_START, end=DATA_END)
prep   = DataPreprocessor()
excess_returns, rf_daily, fred_aligned = prep.prepare(prices, fred)

print('Data loaded. Shape:', excess_returns.shape)

## 1 · Fit Jump Models at Each Rebalancing Date

In [ ]:
from src.models.jump_model import JumpModel
from src.features.return_features import ReturnFeatureBuilder
from src.models.regime_framework import _rebalance_dates

STABILITY_CACHE = RESULTS_DIR / 'stability_jm_fits.pkl'

ret_feat = ReturnFeatureBuilder()

# We use a fixed lambda of 10.0 for this analysis (near median of optimal lambdas)
FIXED_LAMBDA = 10.0

if STABILITY_CACHE.exists():
    print('Loading cached JM fits …')
    with open(STABILITY_CACHE, 'rb') as f:
        jm_fits = pickle.load(f)
    from src.cluster_stability import assert_fits_have_features
    if not assert_fits_have_features(jm_fits, ASSETS):
        print(
            'WARNING:', STABILITY_CACHE,
            'has no feature matrix X. Delete this file and re-run the cell to compute per-refit CVIs.',
        )
else:
    # jm_fits[asset][rebal_date] = {'labels': array, 'dates': DatetimeIndex, 'centroids': array}
    jm_fits = {a: {} for a in ASSETS}
    idx     = excess_returns.index

    rebal_dates = _rebalance_dates(idx, TEST_START, TEST_END, tuple(REBAL_MONTHS))

    for asset in ASSETS:
        print(f'Fitting JM for {asset} across {len(rebal_dates)} windows …')
        X_full = ret_feat.build(excess_returns[asset], asset, for_jm=True)

        for rebal_date in rebal_dates:
            train_start = rebal_date - pd.DateOffset(years=TRAIN_YEARS)
            train_idx   = idx[(idx >= train_start) & (idx < rebal_date)]

            if len(train_idx) < 252:
                continue

            X_train = X_full.reindex(train_idx).dropna()
            if len(X_train) < 100:
                continue

            jm = JumpModel(jump_pen=FIXED_LAMBDA)
            jm.fit(X_train.values)

            # Determine bullish state
            er_train  = excess_returns[asset].reindex(X_train.index)
            stats     = jm.regime_stats(er_train.values)
            bull_st   = max(stats, key=lambda k: stats[k]['mean_daily'])
            labels    = (jm.labels_ != bull_st).astype(int)  # 0=bull, 1=bear

            jm_fits[asset][rebal_date] = {
                'labels':     labels,
                'dates':      X_train.index,
                'centroids':  jm.centroids_,
                'stats':      stats,
                'bull_state': bull_st,
                'X':          np.ascontiguousarray(X_train.values, dtype=np.float64),
            }

    with open(STABILITY_CACHE, 'wb') as f:
        pickle.dump(jm_fits, f)
    print('Saved.')

print('JM fits loaded for all assets and rebalancing dates.')

## 2 · Pairwise stability (consecutive refits) & internal validity (each refit)

In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(16):
    if (_p / "src" / "config" / "settings.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
    if _p.parent == _p:
        raise RuntimeError("Cannot find repo root (src/config/settings.py). cwd=%r" % (Path.cwd(),))
    _p = _p.parent
else:
    raise RuntimeError("Cannot find repo root (src/config/settings.py). cwd=%r" % (Path.cwd(),))

from src.cluster_stability import (
    build_pairwise_stability_df,
    build_internal_validity_df,
    summarize_internal_validity_by_asset,
)

# Pairwise stability between *consecutive* biannual refits (same cadence as λ CV)
stab_df = build_pairwise_stability_df(jm_fits, ASSETS)

# Internal CVIs on the JM training matrix at *each* rebalancing date (geometry; no ground truth)
cvi_df = build_internal_validity_df(jm_fits, ASSETS)
cvi_df.to_csv(RESULTS_DIR / 'stability_cvi_by_rebal.csv', index=False)
cvi_summary = summarize_internal_validity_by_asset(cvi_df).round(4)

print('=== Mean Label Agreement Rate per Asset (consecutive windows) ===')
summary = stab_df.groupby('Asset')[['Agreement', 'ARI']].mean().round(3)
summary.columns = ['Avg Label Agreement', 'Avg ARI']
print(summary.to_string())
summary.to_csv(RESULTS_DIR / 'stability_agreement_summary.csv')

print('\n=== Internal validity (mean over refit dates; ↑Sil & CH/n, ↓DB) ===')
print(cvi_summary.to_string())
cvi_summary.to_csv(RESULTS_DIR / 'stability_cvi_summary_by_asset.csv')

In [ ]:
# Internal CVIs over the sample (one point per biannual JM refit — aligned with λ refresh)
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
assets_plot = ['LargeCap', 'AggBond', 'Gold', 'HighYield']

for ax, col, title, direction in zip(
    axes,
    ['silhouette', 'davies_bouldin', 'calinski_harabasz_per_n'],
    ['Silhouette (↑ better)', 'Davies–Bouldin (↓ better)', 'Calinski–Harabasz / n (↑ better)'],
    ['higher', 'lower', 'higher'],
):
    for asset in assets_plot:
        sub = cvi_df[(cvi_df['Asset'] == asset) & (cvi_df['has_features'])].sort_values('Rebal_date')
        if sub.empty:
            continue
        ax.plot(sub['Rebal_date'], sub[col], marker='o', markersize=3, label=asset, lw=1.2)
    ax.set_ylabel(col.replace('_', ' '))
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.suptitle('Cluster geometry at each refit (same dates as regime / λ calibration)', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'stability_fig4_cvi_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualise agreement over time for key assets
fig, axes = plt.subplots(2, 1, figsize=(13, 8))

for ax, metric in zip(axes, ['Agreement', 'ARI']):
    for asset in ['LargeCap', 'AggBond', 'Gold', 'HighYield']:
        sub = stab_df[stab_df['Asset'] == asset].sort_values('Date_curr')
        ax.plot(sub['Date_curr'], sub[metric], marker='o', markersize=4,
                label=asset, lw=1.2)

    ax.axhline(0.8, color='red', ls='--', lw=0.8, label='0.80 threshold')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} Between Consecutive Window Estimates')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'stability_fig1_agreement_over_time.png', dpi=150)
plt.show()

## 3 · Centroid Drift

In [ ]:
from src.cluster_stability import build_centroid_drift_df

cent_df = build_centroid_drift_df(jm_fits, ASSETS)

print('=== Mean Centroid Drift per Asset (lower = more stable) ===')
drift_summary = cent_df.groupby('Asset')[['Drift_Bull', 'Drift_Bear', 'Drift_Mean']].mean().round(3)
print(drift_summary.to_string())
drift_summary.to_csv(RESULTS_DIR / 'stability_centroid_drift.csv')

## 4 · Regime Occupancy Stability

In [ ]:
from src.cluster_stability import build_occupancy_df

occ_df = build_occupancy_df(jm_fits, ASSETS)

occ_summary = occ_df.groupby('Asset')['Pct_Bear'].agg(['mean', 'std']).round(3)
occ_summary.columns = ['Mean % Bearish', 'Std % Bearish']
print('=== Regime Occupancy (% Bearish) Across Windows ===')
print(occ_summary.to_string())
occ_summary.to_csv(RESULTS_DIR / 'stability_occupancy.csv')

In [ ]:
# Heatmap of % bearish over time
occ_pivot = occ_df.pivot(index='Date', columns='Asset', values='Pct_Bear').reindex(columns=ASSETS)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    occ_pivot.T.astype(float),
    cmap='RdYlGn_r',
    annot=False,
    ax=ax,
    vmin=0, vmax=0.6,
    xticklabels=[str(d.year) if d.month == 1 else '' for d in occ_pivot.index],
    linewidths=0.2,
)
ax.set_title('% Bearish Days per Estimation Window (darker = more bearish)')
ax.set_xlabel('Rebalancing Date')
ax.set_ylabel('Asset')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'stability_fig2_occupancy_heatmap.png', dpi=150)
plt.show()

## 5 · Persistence Ratio

In [ ]:
from src.cluster_stability import build_persistence_df

per_df = build_persistence_df(jm_fits, ASSETS)

per_summary = per_df.groupby('Asset')[['Mean_Run_Length', 'Persistence_Ratio']].mean().round(3)
print('=== Persistence (mean run length across windows) ===')
print(per_summary.to_string())
per_summary.to_csv(RESULTS_DIR / 'stability_persistence.csv')

## 6 · Consolidated Stability Report

In [ ]:
cvi_for_report = cvi_summary.rename(columns={
    'Silhouette (mean)': 'Silhouette',
    'Davies–Bouldin (mean)': 'Davies-Bouldin',
    'Calinski–Harabasz / n (mean)': 'Calinski-Harabasz/n',
})[['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz/n']]

stability_report = (
    summary
    .join(drift_summary[['Drift_Mean']])
    .join(occ_summary[['Std % Bearish']])
    .join(per_summary[['Mean_Run_Length']])
    .join(cvi_for_report)
)
stability_report.columns = [
    'Label Agreement', 'ARI',
    'Centroid Drift', 'Occupancy Std', 'Mean Run Length (days)',
    'Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz/n',
]

print('=== Consolidated Stability Report ===')
print(stability_report.round(3).to_string())
stability_report.to_csv(RESULTS_DIR / 'stability_full_report.csv')

# Normalised bar chart (higher = better geometry / stability)
metrics_norm = stability_report.copy().astype(float)
metrics_norm['Label Agreement'] = metrics_norm['Label Agreement']
metrics_norm['ARI'] = metrics_norm['ARI'].clip(0, 1)
metrics_norm['Centroid Drift'] = 1 / (1 + metrics_norm['Centroid Drift'])
metrics_norm['Occupancy Std'] = 1 / (1 + metrics_norm['Occupancy Std'] * 10)
metrics_norm['Mean Run Length (days)'] = (
    metrics_norm['Mean Run Length (days)'].clip(0, 100) / 100
)
metrics_norm['Silhouette'] = (metrics_norm['Silhouette'].clip(-1, 1) + 1) / 2
metrics_norm['Davies-Bouldin'] = 1 / (1 + metrics_norm['Davies-Bouldin'])
chpn_max = float(np.nanmax(metrics_norm['Calinski-Harabasz/n'].to_numpy()))
if not np.isfinite(chpn_max) or chpn_max == 0:
    chpn_max = 1.0
metrics_norm['Calinski-Harabasz/n'] = metrics_norm['Calinski-Harabasz/n'] / chpn_max

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(ASSETS))
width = 0.09
metric_cols = list(metrics_norm.columns)
colors = plt.cm.tab10.colors

for i, col in enumerate(metric_cols):
    ax.bar(x + i * width, metrics_norm[col], width, label=col, color=colors[i % 10], alpha=0.85)

ax.set_xticks(x + width * (len(metric_cols) - 1) / 2)
ax.set_xticklabels(ASSETS, rotation=35, ha='right')
ax.set_ylabel('Normalised score (↑ = better)')
ax.set_title('Cluster stability & geometry (CVIs averaged over biannual refits)')
ax.legend(fontsize=6, ncol=4, loc='upper right')
ax.grid(alpha=0.3, axis='y')
ax.axhline(0.8, color='red', ls='--', lw=0.7, alpha=0.5)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'stability_fig3_consolidated.png', dpi=150)
plt.show()

## 6b · Macro stress vs pairwise stability (short formal bridge)

Relates **VIX training-window mean** and **HP cycle** (see `src.analysis.time_series_stats`) to **Agreement** between consecutive refits — preview of Hypothesis A (Notebook 04). **Spearman + bootstrap** and **HAC OLS** with a linear time trend.

In [ ]:
from src.analysis.time_series_stats import (
    macro_mean_in_train_window,
    macro_cycle_at_date,
    spearman_bootstrap_ci,
    hac_ols,
)

vix = fred_aligned["vix"].astype(float)
stress_rows = []
for d in sorted(stab_df["Date_curr"].unique()):
    d = pd.Timestamp(d)
    stress_rows.append(
        {
            "Date_curr": d,
            "vix_mean": macro_mean_in_train_window(d, vix, excess_returns.index, TRAIN_YEARS),
            "vix_cycle": macro_cycle_at_date(vix, d),
        }
    )
stress_df = pd.DataFrame(stress_rows)
S = stab_df.merge(stress_df, on="Date_curr", how="left")
S["trend_time"] = (S["Date_curr"] - S["Date_curr"].min()).dt.days / 365.25

lines = []
for asset in ["LargeCap", "AggBond"]:
    sub = S[S["Asset"] == asset].dropna(subset=["Agreement", "vix_cycle", "trend_time"])
    rho, lo, hi = spearman_bootstrap_ci(
        sub["Agreement"].values, sub["vix_cycle"].values, n_boot=1200, seed=7
    )
    lines.append(f"{asset}: Spearman(Agreement, VIX cycle)={rho:.3f} [{lo:.3f},{hi:.3f}] n={len(sub)}")
    res = hac_ols(sub["Agreement"].values, sub[["vix_cycle", "trend_time"]].values)
    lines.append(res.summary().as_text())
report = "\n".join(lines)
print(report)
with open(RESULTS_DIR / "stability_macro_bridge_hac.txt", "w", encoding="utf-8") as f:
    f.write(report)


## 7 · Interpretation

- **High Label Agreement (> 0.80)** across all equity assets suggests that the bullish/bearish regimes are **structurally coherent** rather than window-dependent artefacts.
- **ARI > 0.60** confirms that the improvement over random chance is substantial.
- **Low Centroid Drift** implies that the cluster centres (the representative feature vectors of each regime) are stable across model updates.
- **Internal CVIs over refit dates** (Fig. 4): **Silhouette** and **Davies–Bouldin** are the primary scale-aware geometry metrics; the third panel plots **Calinski–Harabasz / n** (raw CH remains in `stability_cvi_by_rebal.csv` and in Hypothesis B). For links to **macro stress**, see Notebook 04 (only Silhouette and DB are correlated with VIX / STLFSI).
- Assets with **high Occupancy Std** (e.g. Treasury, Gold) suggest more volatile regime classification — consistent with the paper's decision to exclude DD features for those assets in the JM.
- **Long Mean Run Lengths** validate the persistence assumption inherent in the regime definition, justifying the use of regime forecasts in portfolio construction.